
# Projeto MLOps — Detecção de Fraude em Pagamentos Digitais

Notebook reorganizado para ficar mais **limpo, reproduzível e legível**, mantendo a ideia original do projeto e corrigindo problemas do fluxo anterior.

## O que foi melhorado
- organização em etapas claras
- código comentado e padronizado
- correção de inconsistências no feature engineering
- uso de `Pipeline` e `ColumnTransformer`
- avaliação mais adequada para **classe desbalanceada**
- análise de threshold separada da predição padrão

> **Importante:** ajuste o caminho do CSV na célula de configuração antes de rodar.


## 1. Setup e configuração

In [ ]:

# Bibliotecas principais
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [ ]:

# === CONFIGURAÇÃO DO ARQUIVO ===
# Troque pelo caminho correto do seu dataset.
# Exemplo no Colab/Drive:
# DATA_PATH = "/content/drive/MyDrive/Digital_Payment_Fraud_Detection_Dataset.csv"

DATA_PATH = "Digital_Payment_Fraud_Detection_Dataset.csv"

assert Path(DATA_PATH).exists(), (
    f"Arquivo não encontrado: {DATA_PATH}
"
    "Atualize a variável DATA_PATH com o caminho correto antes de continuar."
)


## 2. Carregamento e visão inicial dos dados

In [ ]:

df = pd.read_csv(DATA_PATH)
print(f"Shape inicial: {df.shape}")
display(df.head())


In [ ]:

print("Tipos das colunas:")
display(df.dtypes.to_frame("dtype"))

print("
Informações gerais:")
df.info()


In [ ]:

if "fraud_label" in df.columns:
    class_counts = df["fraud_label"].value_counts().sort_index()
    class_prop = df["fraud_label"].value_counts(normalize=True).sort_index().mul(100)
    resumo_classes = pd.DataFrame({
        "quantidade": class_counts,
        "percentual (%)": class_prop.round(2)
    })
    display(resumo_classes)


## 3. Auditoria de qualidade dos dados

In [ ]:

missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print(f"Linhas duplicadas: {duplicates}")
print("
Colunas com valores ausentes:")
display(missing[missing > 0])


In [ ]:

num_summary = df.describe(include=[np.number]).T
cat_summary = df.describe(include=["object", "category", "bool"]).T

print("Resumo numérico:")
display(num_summary)

print("Resumo categórico:")
display(cat_summary)



## 4. Limpeza e validações básicas

Nesta etapa vamos:
- remover espaços em branco em colunas de texto
- forçar tipos numéricos esperados
- tratar duplicatas
- aplicar validações simples de sanidade


In [ ]:

df_clean = df.copy()

# Remove duplicatas exatas, se existirem
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = len(df_clean)
print(f"Duplicatas removidas: {before - after}")

# Padronização de strings
str_cols = df_clean.select_dtypes(include="object").columns.tolist()
for col in str_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Conversão de colunas numéricas esperadas
expected_numeric = [
    "transaction_amount",
    "account_age_days",
    "transaction_hour",
    "previous_failed_attempts",
    "avg_transaction_amount",
    "is_international",
    "ip_risk_score",
    "login_attempts_last_24h",
    "fraud_label",
]

for col in expected_numeric:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Limpeza inicial concluída.")


In [ ]:

# Validações simples de faixa

def clip_if_exists(frame, column, low=None, high=None):
    if column in frame.columns:
        frame[column] = frame[column].clip(lower=low, upper=high)

# transaction_hour deve estar entre 0 e 23
clip_if_exists(df_clean, "transaction_hour", 0, 23)

# ip_risk_score foi tratado no notebook original como 0..1,
# então mantemos essa convenção e corrigimos o threshold depois.
clip_if_exists(df_clean, "ip_risk_score", 0, 1)

# Colunas binárias -> 0/1
for col in ["is_international", "fraud_label"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0).astype(int).clip(0, 1)

# Valores não positivos em montantes viram NaN para imputação posterior
for col in ["transaction_amount", "avg_transaction_amount"]:
    if col in df_clean.columns:
        invalid = (df_clean[col] <= 0).sum()
        if invalid > 0:
            print(f"{col}: {invalid} valores não positivos convertidos em NaN")
            df_clean.loc[df_clean[col] <= 0, col] = np.nan

# account_age_days não pode ser negativo
if "account_age_days" in df_clean.columns:
    negatives = (df_clean["account_age_days"] < 0).sum()
    if negatives > 0:
        print(f"account_age_days: {negatives} valores negativos convertidos em NaN")
        df_clean.loc[df_clean["account_age_days"] < 0, "account_age_days"] = np.nan

print("Validações concluídas.")


## 5. Feature engineering

In [ ]:

# Importante: daqui em diante usamos df_clean (e não o df original)
df_feat = df_clean.copy()

# Razão entre valor da transação e média histórica do usuário
if {"transaction_amount", "avg_transaction_amount"}.issubset(df_feat.columns):
    df_feat["amount_vs_avg"] = df_feat["transaction_amount"] / (df_feat["avg_transaction_amount"] + 1)
    df_feat["high_value_transaction"] = (
        df_feat["transaction_amount"] > 2 * df_feat["avg_transaction_amount"]
    ).astype(int)

# Transação noturna
if "transaction_hour" in df_feat.columns:
    df_feat["night_transaction"] = df_feat["transaction_hour"].between(0, 5).astype(int)

# Score comportamental simples
if {"login_attempts_last_24h", "previous_failed_attempts"}.issubset(df_feat.columns):
    df_feat["behavior_risk_score"] = (
        df_feat["login_attempts_last_24h"] + df_feat["previous_failed_attempts"]
    )

# Conta nova
if "account_age_days" in df_feat.columns:
    df_feat["new_account"] = (df_feat["account_age_days"] < 30).astype(int)

# Correção importante do notebook original:
# se ip_risk_score está em 0..1, o threshold correto não é > 70, e sim > 0.70
if {"is_international", "ip_risk_score"}.issubset(df_feat.columns):
    df_feat["international_high_risk"] = (
        (df_feat["is_international"] == 1) & (df_feat["ip_risk_score"] > 0.70)
    ).astype(int)

# Muito login + transação acima da média
if {"login_attempts_last_24h", "transaction_amount", "avg_transaction_amount"}.issubset(df_feat.columns):
    df_feat["high_activity_risk"] = (
        (df_feat["login_attempts_last_24h"] > 5) &
        (df_feat["transaction_amount"] > df_feat["avg_transaction_amount"])
    ).astype(int)

print("Shape após feature engineering:", df_feat.shape)
display(df_feat.head())


## 6. Definição de variáveis de entrada e alvo

In [ ]:

TARGET = "fraud_label"
DROP_COLUMNS = ["transaction_id", "user_id"]  # IDs não agregam sinal preditivo útil aqui

assert TARGET in df_feat.columns, f"A coluna alvo '{TARGET}' não foi encontrada."

X = df_feat.drop(columns=[c for c in DROP_COLUMNS + [TARGET] if c in df_feat.columns])
y = df_feat[TARGET].copy()

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)
print("Distribuição do alvo:")
display(y.value_counts(normalize=True).rename("proporcao"))


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape, y_test.shape)


## 7. Pré-processamento com Pipeline

In [ ]:

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Features numéricas:", numeric_features)
print("
Features categóricas:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## 8. Modelo

In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf_model),
])

model.fit(X_train, y_train)
print("Modelo treinado com sucesso.")



## 9. Avaliação do modelo

Como fraude costuma ser uma classe minoritária, além de ROC-AUC, também avaliamos:
- **Average Precision (PR-AUC)**
- relatório de classificação
- matriz de confusão


In [ ]:

y_prob = model.predict_proba(X_test)[:, 1]
y_pred_default = model.predict(X_test)

roc_auc = roc_auc_score(y_test, y_prob)
ap_score = average_precision_score(y_test, y_prob)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average Precision (PR-AUC): {ap_score:.4f}")

print("
Relatório de classificação (threshold padrão = 0.50):")
print(classification_report(y_test, y_pred_default, digits=4))


In [ ]:

cm = confusion_matrix(y_test, y_pred_default)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, colorbar=False)
ax.set_title("Matriz de confusão — threshold padrão (0.50)")
plt.show()


In [ ]:

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.show()


In [ ]:

# Curva Precision-Recall
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f"AP = {ap_score:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")
plt.legend()
plt.show()


## 10. Análise de threshold

In [ ]:

# Em problemas de fraude, muitas vezes um threshold menor que 0.50 captura mais fraudes.
threshold_grid = np.arange(0.01, 0.31, 0.01)

threshold_results = []
for t in threshold_grid:
    pred_t = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_t = tp / (tp + fn) if (tp + fn) > 0 else 0
    threshold_results.append({
        "threshold": round(float(t), 2),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": precision_t,
        "recall": recall_t,
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.head(10))


In [ ]:

# Exemplo: escolher o threshold com maior recall mantendo alguma precisão mínima.
# Você pode adaptar o critério conforme o objetivo do negócio.
filtered = threshold_df[threshold_df["precision"] >= 0.10].copy()
filtered = filtered.sort_values(["recall", "precision"], ascending=False)
display(filtered.head(10))


In [ ]:

# Ajuste manual de threshold para inspeção
chosen_threshold = 0.10
y_pred_custom = (y_prob >= chosen_threshold).astype(int)

print(f"Threshold escolhido: {chosen_threshold:.2f}
")
print(classification_report(y_test, y_pred_custom, digits=4))

cm_custom = confusion_matrix(y_test, y_pred_custom)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm_custom).plot(ax=ax, colorbar=False)
ax.set_title(f"Matriz de confusão — threshold {chosen_threshold:.2f}")
plt.show()


## 11. Importância das features

In [ ]:

# Para extrair os nomes finais após o pré-processamento
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

feat_importance = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feat_importance.head(15))


In [ ]:

plot_df = feat_importance.head(15).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.title("Top 15 features mais importantes")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()



## 12. Conclusões

### Principais correções em relação ao notebook original
- o feature engineering agora é feito em `df_clean` / `df_feat`, e não parcialmente no `df` original
- o threshold de `international_high_risk` foi corrigido de `> 70` para `> 0.70`
- a matriz de confusão agora depende de imports explícitos e de uma etapa de predição consistente
- o fluxo de pré-processamento está encapsulado em pipeline, reduzindo risco de inconsistência entre treino e teste

### Próximos passos sugeridos
- testar `XGBoost`, `LightGBM` ou `HistGradientBoosting`
- usar validação cruzada estratificada
- comparar com um baseline simples
- calibrar probabilidades do modelo
- transformar este notebook em pipeline de produção com scripts separados (`src/`, `train.py`, `predict.py`, etc.)
